Traverse all folders under `dataset4geodiff\out_openai_sematic_geodiff_txt`, then iterate through the first-level folders. Each folder name is the APK name.

Inside each folder:
- Find the JSON file ending with `_threshold_mapping` and extract `path_mapping_results.all_kept_mappings.ours_id`, which is the clause ID.
- Find the JSON file ending with `_processed_converted` and extract `app_info.baseline_version` and `app_info.target_version`.
- In `dataset4geodiff/raw_geodiff_txt/{apkname}`, find the CSV file ending with `_country_versions`. It contains the mappings between `baseline_version` and regions, and between `target_version` and regions.

Build an empty DataFrame whose rows are clause IDs and columns are regions. If a clause ID appears in the data above, set the value to 1 for the countries corresponding to `target_version` and set all other values to 0.

Because the current JSON does not contain a field for region-specific decisions, the 11 region columns must temporarily receive the same value: if a clause appears and its value is 1, set all 11 columns to 1; otherwise set them to 0. This interpretation matches the structure of the provided files.

REGIONS = [
    "California", "Texas", "Germany", "Turkey", "Egypt",
    "Vietnam", "Nigeria", "India", "Saudi", "Bangladesh", "Pakistan"
]

In [ ]:
# B in : first_2,   first_third, first_4

In [11]:
# {f_position}\{s_position}
f_position = "AA1_first_batch"
s_position = "AA4_forth_100_batch" # AA1_first_328_batch, AA4_forth_100_batch, AA6_sisth_74_batch  ## AA1_first_328_batch, AA4_forth_100_batch, AA6_sisth_74_batch

# {parameter}
# parameter first_1,  first_2,   first_third, first_4, first_5, first_6
# second_1, second_2, second_3, second_4, second_5, second_6, second_7
# third_1,  third_2,  third_3,  third_4,  third_5,  third_6,  third_7
# fourth_1
parameter = "first_6"

In [ ]:
from pathlib import Path
import json
import pandas as pd

# first_1,  first_2,   first_third, first_4, first_5, first_6
# second_1, second_2, second_3, second_4, second_5, second_6, second_7
# third_1,  third_2,  third_3,  third_4,  third_5,  third_6,  third_7
# fourth_1

ROOT_DIR = Path(r"dataset4geodiff/apk_versions_long.csv")
OUT_ROOT = Path(rf"dataset4geodiff/out_openai_sematic_geodiff_txt/{parameter}")
# RAW_ROOT = Path(r"dataset4geodiff/raw_geodiff_txt")
TEMPLATE_XLSX = Path(r"dataset4geodiff\can_detect_11regions_regulations_geodiff.xlsx")

# True: set all regions in the template to 1 when a clause appears; False: set 1 only for the regions corresponding to target_version
USE_UNIFORM_REGION_FILL = False

In [13]:
def load_matrix_template_axes(template_xlsx_path: Path):
    tpl = pd.read_excel(template_xlsx_path, dtype=str)
    if tpl.empty:
        raise ValueError(f"Template xlsx is empty: {template_xlsx_path}")

    id_col = "Clause" if "Clause" in tpl.columns else tpl.columns[0]
    print(f"ID Column: {id_col}")
    clause_axis = [
        str(x).strip()
        for x in tpl[id_col].tolist()
        if str(x).strip() and str(x).strip().lower() != "nan"
    ]
    region_axis = [str(c).strip() for c in tpl.columns if c != id_col]

    if not clause_axis:
        raise ValueError(f"No clause rows found in template: {template_xlsx_path}")
    if not region_axis:
        raise ValueError(f"No region columns found in template: {template_xlsx_path}")
    print(f"Clause Axis: {clause_axis}\n", "len(clause_axis):", len(clause_axis))
    print(f"Region Axis: {region_axis}\n", "len(region_axis):", len(region_axis))
    return clause_axis, region_axis


TEMPLATE_CLAUSE_AXIS, TEMPLATE_REGION_AXIS = load_matrix_template_axes(TEMPLATE_XLSX)
REGIONS = TEMPLATE_REGION_AXIS
print(REGIONS)

ID Column: Clause
Clause Axis: ['P1', 'P2', 'P3', 'P4', 'P5', 'P6', 'P7', 'P8', 'P10', 'P11', 'P12', 'P13', 'P14', 'P15', 'P16', 'P17', 'P18', 'P19', 'P20', 'P21', 'CR1', 'CR2', 'CR3', 'CR4', 'CR6', 'C1', 'R1', 'R2', 'R4', 'R5', 'R6', 'R7', 'R8', 'R9', 'R10', 'R11', 'R13', 'R14', 'R15', 'R16', 'R17', 'R19', 'R20', 'E1', 'E24', 'E25', 'E26', 'E27', 'E28', 'O3']
 len(clause_axis): 50
Region Axis: ['California', 'Texas', 'Germany', 'Turkey', 'Egypt', 'Vietnam', 'Nigeria', 'India', 'Saudi', 'Bangladesh', 'Pakistan']
 len(region_axis): 11
['California', 'Texas', 'Germany', 'Turkey', 'Egypt', 'Vietnam', 'Nigeria', 'India', 'Saudi', 'Bangladesh', 'Pakistan']


In [11]:
# # find all versions of each apk in the dataset
# apk_version_df = pd.read_csv("dataset4geodiff/apk_versions_long.csv")
# apk_version_df.head(), apk_version_df.shape

# example: find all versions of HinKhoj.Dictionary
# for version in apk_version_df[apk_version_df['apkname'] == 'HinKhoj.Dictionary']["version"].values:
#     print(str(version).strip())

(                    apkname  version
 0        HinKhoj.Dictionary      310
 1        HinKhoj.Dictionary      320
 2  ae.brandsforless.android      412
 3  ae.brandsforless.android      413
 4       air.bg.lan.Monopoli  7000009,
 (2193, 2))

In [ ]:
def process_version(apk_version_df, apkname, baseline_version):
    apk_rows = apk_version_df[apk_version_df['apkname'] == apkname]
    if apk_rows.empty:
        print(f"[WARN] {apkname} not found in version mapping file")
        return None
    versions = apk_rows["version"].values
    for v in versions:
        if str(v).strip()==baseline_version:
            continue
        target_version = str(v).strip()
    if not target_version:
        print(f"[WARN] {apkname} has empty version in mapping file")
        return None

    return target_version

def find_one_file(folder: Path, pattern: str):
    matches = sorted(folder.glob(pattern))
    return matches[0] if matches else None


def _extract_mapping_items(obj):
    if isinstance(obj, dict):
        all_kept = obj.get("all_kept_mappings", [])
        if isinstance(all_kept, list):
            return all_kept
        return []
    if isinstance(obj, list):
        items = []
        for x in obj:
            if isinstance(x, dict):
                if "all_kept_mappings" in x:
                    v = x.get("all_kept_mappings", [])
                    if isinstance(v, list):
                        items.extend(v)
                else:
                    items.append(x)
        return items
    return []




def load_clause_ids(threshold_mapping_path: Path):
    with open(threshold_mapping_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    items = []
    if isinstance(data, dict):
        pmr = data.get("path_mapping_results", {})
        items = _extract_mapping_items(pmr)
    elif isinstance(data, list):
        for entry in data:
            if not isinstance(entry, dict):
                continue
            if "path_mapping_results" in entry:
                items.extend(_extract_mapping_items(entry.get("path_mapping_results")))
            else:
                items.extend(_extract_mapping_items(entry))

    clause_ids = []
    for item in items:
        if not isinstance(item, dict):
            continue
        cid = str(item.get("ours_id", "")).strip()
        if cid:
            clause_ids.append(cid)
    return sorted(set(clause_ids))


def load_versions_1(processed_converted_path: Path):
    with open(processed_converted_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    app_info = data.get("app_info", {})
    baseline_version = str(app_info.get("baseline_version", "")).strip()
    target_version = str(app_info.get("target_version", "")).strip()
    return baseline_version, target_version

def find_target_version(all_info_df: pd.DataFrame, app_id: str, baseline: str) -> str:
    # Find a version different from baseline among all APK version records.
    apk = all_info_df[all_info_df["apkname"] == app_id]
    if apk.empty:
        return ""
    vals = apk["version"].astype(str).str.strip().unique().tolist()
    uniq = []
    for v in vals:
        if v not in uniq:
            if v != baseline:
                uniq.append(v)
    print(app_id,uniq)
    return uniq[0] if uniq else ""

def load_versions(ROOT_DIR: Path, country_csv_path: Path, apkname: str):
    all_apks_info = pd.read_csv(ROOT_DIR, dtype=str)
    # 1) Read country_versions.csv.
    try:
        df = pd.read_csv(country_csv_path, dtype=str).fillna("")
    except Exception as e:
        print(f"[SKIP] {country_csv_path}: failed to read CSV -> {e}")
        return "", ""

    if df.empty:
        print(f"[SKIP] {country_csv_path}: CSV is empty")
        return "", ""

    row = df.iloc[0]
    baseline_version = str(row.get("usa_version_code", "")).strip()

    target_version = find_target_version(all_apks_info, apkname, baseline_version)

    return baseline_version, target_version

REGION_COLUMN_ALIASES = {
    "California": ["usa_version_code"],
    "Texas": ["usa_version_code"],
    "Germany": ["germany_version_code"],
    "Turkey": ["turkey_version_code"],
    "Egypt": ["egypt_version_code"],
    "Vietnam": ["vietnam_version_code"],
    "Nigeria": ["nigeria_version_code"],
    "India": ["india_version_code"],
    "Saudi": ["saudi_version_code"],
    "Bangladesh": ["bangladesh_version_code"],
    "Pakistan": ["pakistan_version_code"],
}


def get_region_candidate_columns(region: str):
    aliases = REGION_COLUMN_ALIASES.get(region, [])
    default_col = f"{region.lower()}_version_code"
    # print(f"Region '{region}' candidate columns: {aliases + [default_col]}")
    return list(dict.fromkeys(aliases + [default_col]))


def regions_for_version(country_df: pd.DataFrame, version: str):
    target_regions = []
    v = str(version).strip()
    if not v:
        return target_regions

    for region in REGIONS:
        candidate_cols = get_region_candidate_columns(region)
        matched = False
        for col in candidate_cols:
            if col not in country_df.columns:
                continue
            series = country_df[col].astype(str).str.strip()
            if (series == v).any():
                matched = True
                break
        if matched:
            target_regions.append(region)
    return target_regions


def build_matrix(detected_clause_ids, active_regions):
    matrix = pd.DataFrame(0, index=TEMPLATE_CLAUSE_AXIS, columns=TEMPLATE_REGION_AXIS, dtype=int)
    matrix.index.name = "Clause" # "clause_id"

    clause_set = set(detected_clause_ids)
    region_set = set(active_regions)

    for cid in TEMPLATE_CLAUSE_AXIS:
        if cid not in clause_set:
            continue
        for region in TEMPLATE_REGION_AXIS:
            if region in region_set:
                matrix.at[cid, region] = 1
    return matrix


def process_one_apk_folder(apk_folder: Path):
    apkname = apk_folder.name

    

    threshold_mapping_path = find_one_file(apk_folder, "*process.json")
    if threshold_mapping_path is None:
        threshold_mapping_path = find_one_file(apk_folder, "*process*.json")

    # processed_converted_path = find_one_file(apk_folder, "*_processed_converted.json")

    raw_apk_folder = OUT_ROOT / apkname
    country_csv_path = find_one_file(raw_apk_folder, "*_country_versions.csv")

    print("threshold_mapping_path", threshold_mapping_path)
    if threshold_mapping_path is None:
        print(f"[SKIP] {apkname}: threshold mapping json not found")
        return
    # if processed_converted_path is None:
    #     print(f"[SKIP] {apkname}: processed_converted json not found")
    #     return
    if country_csv_path is None:
        print(f"[SKIP] {apkname}: country_versions csv not found")
        return

    clause_ids = load_clause_ids(threshold_mapping_path)
    print(f"[INFO] {apkname}: detected clause ids: {clause_ids}")
    baseline_version, target_version = load_versions(ROOT_DIR, country_csv_path, apkname)
    print(f"[INFO] {apkname}: US version is baseline_version={baseline_version}, target_version={target_version}")

    country_df = pd.read_csv(country_csv_path, dtype=str)
    country_df.drop(columns=['package_name'], inplace=True)
    # print(f"{country_df.head()}")



    target_regions = regions_for_version(country_df, target_version)
    # print(apkname, target_regions)

    if USE_UNIFORM_REGION_FILL:
        active_regions = TEMPLATE_REGION_AXIS if clause_ids else []
    else:
        active_regions = target_regions

    print("active_regions", active_regions)

    matrix_df = build_matrix(clause_ids, active_regions)
    # Remove the row with index "P9".
    # matrix_df = matrix_df.drop("P9")
    print(matrix_df.shape)


    out_path = apk_folder / f"{apkname}_clause_region_matrix.csv"

    matrix_df.to_csv(out_path, encoding="utf-8-sig")

    print(
        f"[OK] {apkname}: clauses_detected={len(clause_ids)}, "
        f"template_rows={len(TEMPLATE_CLAUSE_AXIS)}, template_cols={len(TEMPLATE_REGION_AXIS)}, "
        f"target_version={target_version}, target_regions={target_regions}, saved={out_path}"
    )

In [ ]:
# # Batch-iterate through the first-level APK folders.
for apk_folder in sorted([p for p in OUT_ROOT.iterdir() if p.is_dir()]):
    apkname = apk_folder.name
    # skip folders that are not APK package names
    if "." not in apkname:
        print(f"[SKIP] Not an APK folder: {apkname}")
        continue

    print(f"Processing APK folder: {apk_folder}")
    process_one_apk_folder(apk_folder)
# com.benoitletondor.pixelminimalwatchface
# process_one_apk_folder(Path(r"dataset4geodiff\out_openai_sematic_geodiff_txt\air.com.bigwigmedia.hotdogbush")) # com.bandagames.mpuzzle.gp ae.brandsforless.android  air.bg.lan.Monopoli  air.com.bigwigmedia.hotdogbush